# CST／CRP 分步处理

默认用合成 DWI 网格与模拟流线练习整个离线入口。正式数据需要已审核的校正 DWI、脑域、配准链和图谱标签。运行至每个审核点，先检查本阶段输出，再执行下一格。

- 练习用默认设置。
- 分析现有追踪结果时填写 REPLAY_CONFIG，登记父 TCK、播种坐标日志和计数。
- 复查已生成的筛选结果时填写 EXISTING_RESULTS。这个入口读取筛选产物并可重新生成 HTML。
- MRtrix 3.0.4 的拟合与追踪在已有本地环境中按需开启。详见 [操作与审核](../docs/TRACTOGRAPHY.md)。

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import tempfile

cwd = Path.cwd()
repo = next((p for p in (cwd, *cwd.parents) if (p / "run_tractography.py").is_file()), None)
if repo is None:
    raise RuntimeError("请从 NEO_MRI 仓库或其 notebooks 目录打开。")
sys.path.insert(0, str(repo))
sys.path.insert(0, str(repo / "code"))

from IPython.display import FileLink, display
from tractography.viewer import interactive

%matplotlib inline

def cli(stage, output, **options):
    # 使用当前内核的 Python，并直接调用同一套命令行后端。
    argv = [sys.executable, "-B", str(repo / "run_tractography.py"), stage, "--output", str(output)]
    for name, value in options.items():
        argv += [f"--{name}", str(value)]
    result = subprocess.run(argv, cwd=repo, capture_output=True, encoding="utf-8",
                            env={**os.environ, "PYTHONIOENCODING": "utf-8", "MPLBACKEND": "Agg"})
    if result.returncode:
        raise RuntimeError(result.stderr or result.stdout)
    return json.loads(result.stdout)

def link(path):
    path = Path(path).resolve()
    print(path)
    if path.is_relative_to(repo):
        display(FileLink(Path("..") / path.relative_to(repo) if cwd == repo / "notebooks"
                         else path.relative_to(repo)))

# 演示输出留在检出目录下的 work，每次执行生成新的目录。
work = repo / "work"
work.mkdir(exist_ok=True)

In [ ]:
# 真实输入配置使用文件系统路径。所有未用的入口保留 None。
FIT_CONFIG = None
TRACK_CONFIG = None
REPLAY_CONFIG = None
EXISTING_RESULTS = None
OUTPUT_PARENT = None    # 真实数据指定该访视的 results/DWI_CST 目录
RUN_NATIVE_FIT = False
RUN_NATIVE_TRACK = False

selected_modes = sum(x is not None for x in (REPLAY_CONFIG, EXISTING_RESULTS))
if selected_modes > 1 or (selected_modes and RUN_NATIVE_TRACK):
    raise ValueError("现有结果、已有父流线、原生追踪三个入口请选择一个。")
if (RUN_NATIVE_FIT and not FIT_CONFIG) or (RUN_NATIVE_TRACK and not TRACK_CONFIG):
    raise ValueError("开启原生阶段前请提供对应配置。")
if any((FIT_CONFIG, TRACK_CONFIG, REPLAY_CONFIG, EXISTING_RESULTS)) and OUTPUT_PARENT is None:
    raise ValueError("正式数据请指定访视下的 OUTPUT_PARENT。")
output_parent = Path(OUTPUT_PARENT).resolve() if OUTPUT_PARENT else work
output_parent.mkdir(parents=True, exist_ok=True)
output = Path(tempfile.mkdtemp(prefix="tractography_", dir=output_parent))
print(output)

## 1. 核对输入与命令

先核对校正后的 b0、模型输入、种子和追踪脑域是否来自同一次访视。查看脑干与延髓覆盖情况，确认图谱标签已审核并映射到 DWI 网格。拟合计划读取配置与输入哈希。原生执行还会检查模型网格和梯度。

In [ ]:
if FIT_CONFIG:
    display(cli("fit-plan", output / "fit", config=FIT_CONFIG))
else:
    print("本次跳过扩散模型拟合计划。")

## 2. 按需拟合扩散模型

张量分支生成 FA、MD、AD、RD。单组织 CSD 分支估计白质纤维方向分布。检查 FA 和 FOD 是否覆盖预期白质与脑干，以及高低壳的 b 值和方向是否按采集方案准备。追踪配置要登记实际拟合产物的路径和哈希。

In [ ]:
if RUN_NATIVE_FIT:
    display(cli("fit", output / "fit", config=FIT_CONFIG))
else:
    print("使用已有拟合产物或直接进入筛选。")

## 3. 按需生成父流线

先检查命令里的模型、起点、侧别、播种尝试预算、随机种子和追踪脑域。确认后执行下一格。原生追踪阶段保存父 TCK、播种坐标、执行日志和供下一步使用的配置。

In [ ]:
if TRACK_CONFIG:
    display(cli("track-plan", output / "tracking", config=TRACK_CONFIG))
else:
    print("本次跳过原生追踪计划。")

In [ ]:
tracked = None
if RUN_NATIVE_TRACK:
    tracked = cli("track", output / "tracking", config=TRACK_CONFIG)
    display(tracked)

## 4. 重放筛选并生成 HTML

后端读取播种坐标，按皮层起点、向下走向和有序解剖门筛选。每个门保存流线索引、访问计数和逐层 15% 核心区域。核对每组的分母和分子，再打开标准 HTML。中途有错误时，本次诊断留在原目录，重试另建目录。

In [ ]:
if EXISTING_RESULTS:
    results = Path(EXISTING_RESULTS).resolve(strict=True)
elif tracked or REPLAY_CONFIG:
    replay_config = tracked["replay_config"] if tracked else REPLAY_CONFIG
    results = output / "screened"
    display(cli("screen", results, config=replay_config))
elif FIT_CONFIG or TRACK_CONFIG:
    results = None
    print("计划阶段结束。准备好父流线、播种日志及配置后可单独执行筛选。")
else:
    example = cli("demo", output / "synthetic")
    results = Path(example["results"])

if results is not None:
    if EXISTING_RESULTS or tracked or REPLAY_CONFIG:
        report_path = Path(cli("report", output / "CST_CRP_report.html", results=results)["output"])
    else:
        report_path = Path(example["report"])
    link(report_path)

## 5. 交互审核与按门导出

选择模型、起点、侧别、运行与筛选门，再点击「查看」。对照父流线、筛选后流线、覆盖计数和核心图。三维图最多显示前 120 条供浏览，三个正交切面展示 b0 上的覆盖位置。重点检查皮层起点、内囊、脑脚和脑桥路径，以及与追踪脑域边缘的关系。

需要在 ITK-SNAP 或 3D Slicer 继续审核时，可用下一格导出所选门的索引、NIfTI 和筛选流线。

In [ ]:
controls = interactive(results) if results is not None else None

In [ ]:
EXPORT_CURRENT = False
if EXPORT_CURRENT:
    if controls is None:
        raise RuntimeError("请先完成筛选并打开交互控件。")
    run = controls["run"].value
    gate = controls["gate"].value
    export_dir = Path(tempfile.mkdtemp(prefix="export_", dir=output)) / "files"
    display(cli("export", export_dir, results=results, run=run, gate=gate))